In [6]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, FunctionTransformer
from sklearn import set_config
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer

data = pd.read_csv('train.csv', index_col='Id')
set_config(transform_output='pandas')

X = data.drop('SalePrice', axis=1)
y = data.SalePrice

In [7]:
class BsmtDetailsReplacer(BaseEstimator, TransformerMixin):
    def __init__(self, BsmtCond):
        self.BsmtCond = BsmtCond
        self.y = None
    def fit(self, X, y=None):        
        return self
    def transform(self, X):
        for col in X.columns:
            X.loc[(self.BsmtCond.notna()) & (X[col].isna()), col] = X[col].mode()[0]
        return X

In [8]:
drop_cols = ['Alley', 'RoofStyle', 'Exterior2nd', 'MoSold']
Bsmt_details_cols = ['BsmtFinType1', 'BsmtFinType2']
impute_zero_cols = ['BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath', 'GarageYrBlt', 'MasVnrArea']
impute_No_cols = ['BsmtQual', 'BsmtCond', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature']
impute_mean_cols = ['LotFrontage', 'GarageCars', 'GarageArea']
impute_mode_cols = ['Electrical', 'Utilities', 'KitchenQual', 'Functional']

BsmtExposure_pl = Pipeline(steps=[
    ('impute', BsmtDetailsReplacer(X['BsmtCond'])),
    ('fill_na', SimpleImputer(strategy='constant', fill_value='NB'))
])
BsmtDetails_pl = Pipeline(steps=[
    ('impute', BsmtDetailsReplacer(X['BsmtCond'])),
    ('fill_na', SimpleImputer(strategy='constant', fill_value='No'))
])

preprocessing_ct = ColumnTransformer(transformers=[
    ('BsmtExposure_prepro', BsmtExposure_pl, ['BsmtExposure']),
    ('BsmtDetails_prepro', BsmtDetails_pl, Bsmt_details_cols),
    ('impute_zero', SimpleImputer(strategy='constant', fill_value=0), impute_zero_cols),
    ('impute_No', SimpleImputer(strategy='constant', fill_value='No'), impute_No_cols),
    ('impute_mean', SimpleImputer(strategy='mean'), impute_mean_cols),
    ('impute_mode', SimpleImputer(strategy='most_frequent'), impute_mode_cols),
    ('drop', 'drop', drop_cols)
], remainder='passthrough', verbose_feature_names_out=False)


In [9]:
def add_cols_naming(transformer_name, feature_name):
    if (transformer_name == 'keep') | (transformer_name == 'remainder'):
        name = feature_name
    else:
        name = transformer_name
    return name

add_ct = ColumnTransformer(transformers=[
    ('HasGarage', FunctionTransformer(lambda x: x != 0), ['GarageYrBlt']),
    ('HasBsmt', FunctionTransformer(lambda x: x != 'No'), ['BsmtQual']),
    ('HasPool', FunctionTransformer(lambda x: x != 0), ['PoolArea']),
    ('keep', 'passthrough', ['GarageYrBlt', 'BsmtQual', 'PoolArea']),
], remainder='passthrough', verbose_feature_names_out=add_cols_naming)

In [10]:
Ordinal_cols = ['LotShape', 'Utilities', 'LandSlope', 'ExterQual',
                'ExterCond', 'BsmtQual', 'BsmtCond', 'BsmtExposure',
                'BsmtFinType1', 'BsmtFinType2', 'HeatingQC', 'KitchenQual',
                'Functional', 'FireplaceQu', 'GarageFinish', 'GarageQual',
                'GarageCond', 'PavedDrive', 'PoolQC', 'Fence']
LotShape_cats = ['Reg','IR1','IR2','IR3']
Utilities_cats = ['ELO', 'NoSeWa', 'NoSewr', 'AllPub']
LandSlope_cats = ['Gtl', 'Mod', 'Sev']
ExterQual_cats = ['Po', 'Fa', 'TA', 'Gd', 'Ex']
BsmtQual_cats = ['Ex', 'Gd', 'TA', 'Fa', 'Po', 'No']
BsmtExposure_cats = ['Gd', 'Av', 'Mn', 'No', 'NB']
BsmtFinType1_cats = ['No', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ']
Functional_cats = ['Sal', 'Sev', 'Maj2', 'Maj1', 'Mod', 'Min2', 'Min1', 'Typ']
GarageFinish_cats = ['No', 'Unf', 'RFn', 'Fin']
PavedDrive_cats = ['Y', 'P', 'N']
PoolQC_cats = ['No', 'Fa', 'TA', 'Gd', 'Ex']
Fence_cats = ['No', 'MnWw', 'GdWo', 'MnPrv', 'GdPrv']

Order_encoder = OrdinalEncoder(categories=[LotShape_cats, Utilities_cats, LandSlope_cats, ExterQual_cats,
                                           ExterQual_cats, BsmtQual_cats, BsmtQual_cats, BsmtExposure_cats,
                                           BsmtFinType1_cats, BsmtFinType1_cats, ExterQual_cats, ExterQual_cats,
                                           Functional_cats, BsmtQual_cats, GarageFinish_cats, BsmtQual_cats,
                                           BsmtQual_cats, PavedDrive_cats, PoolQC_cats, Fence_cats],
                                           encoded_missing_value=-1)

In [11]:
OH_cols = ['MSZoning', 'Street', 'LandContour', 'LotConfig', 'Neighborhood', 'Condition1', 'Condition2',
           'BldgType', 'HouseStyle', 'RoofMatl', 'Exterior1st', 'MasVnrType', 'Foundation', 'Heating',
           'CentralAir', 'Electrical', 'GarageType', 'MiscFeature', 'SaleType', 'SaleCondition']
OH_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

In [12]:
encode_ct = ColumnTransformer(transformers=[
    ('ordered', Order_encoder, Ordinal_cols),
    ('onehot', OH_encoder, OH_cols)
], remainder='passthrough')

In [ ]:
forest_model = RandomForestRegressor(n_estimators=350, max_depth=28, max_features=0.15, random_state=0)

# xgb_model = xgb.XGBRegressor(n_estimators=250, max_depth=20, learning_rate=0.05, n_jobs=2, early_stopping_rounds=5)
# xgb_preproc_pl = Pipeline(steps=[
#     ('preproc', preprocessing_ct),
#     ('add_cols', add_ct),
#     ('encode', encode_ct)
# ])

model_pl = Pipeline(steps=[
    ('preproc', preprocessing_ct),
    ('add_cols', add_ct),
    ('encode', encode_ct),
    ('model', forest_model)
])

# X_train, X_val, y_train, y_val = train_test_split(X, y, train_size=0.8, test_size=0.2, random_state=0)

# xgb_preproc_pl.fit(X_train, y_train)
# X_val_xgb = xgb_preproc_pl.transform(X_val)
# model_pl.fit(X_train, y_train, model__eval_set=[(X_val_xgb, y_val)], model__verbose=False)

model_pl.fit(X_train, y_train)

# pred_val = model_pl.predict(X_val)
# print("MAE (Your approach):")
# print(mean_absolute_error(y_val, pred_val))

# def score_per_param(param):
#     param_pl = Pipeline(steps=[
#         ('preproc', preprocessing_ct),
#         ('add_cols', add_ct),
#         ('encode', encode_ct),
#         ('model', RandomForestRegressor(n_estimators=350, max_depth=28, max_features=param, random_state=0))
#     ])
#     return (-1 * cross_val_score(param_pl, X, y, scoring='neg_mean_absolute_error')).mean()
# {n : score_per_param(n) for n in np.arange(0.1, 0.45, 0.05)}

MAE (Your approach):
17159.955078125


In [17]:
X_test = pd.read_csv('test.csv', index_col='Id')

pred_test = model_pl.predict(X_test)

In [18]:
output = pd.DataFrame({'Id': X_test.index, 'SalePrice': pred_test})
output.to_csv('submission.csv', index=False)